# Día 16 — Reinforcement Learning, Sim2Real y agentes que aprenden actuando

En este cuaderno vamos a ver **ideas básicas de aprendizaje por refuerzo** usando entornos pequeños de `Gymnasium`.

La meta no es convertirnos en expertos en RL en una sesión. La meta es entender el cambio de paradigma:

> En aprendizaje por refuerzo, el modelo no solo predice: **actúa**, observa consecuencias y mejora una **política**.

Trabajaremos con tres casos prácticos. En cada uno compararemos el mismo patrón:

1. **Política aleatoria:** sirve como línea base. Si esto ya funciona, el problema era demasiado fácil.
2. **Política heurística:** usa intuición humana o control clásico simple. Ayuda a pensar qué variables importan.
3. **Política aprendida:** ajusta una política usando recompensa. No siempre será perfecta, pero muestra cómo aparece el aprendizaje por interacción.

Los casos serán:

1. **Demo guiada:** un entorno tipo CartPole en MuJoCo: `InvertedPendulum-v5`. Si MuJoCo no está disponible, el cuaderno usa `CartPole-v1` como respaldo.
2. **Ejercicio 1:** `MountainCar-v0`, para entender recompensa diferida y exploración.
3. **Ejercicio 2:** `LunarLander-v3`, para comparar control intuitivo, ajuste de parámetros y aprendizaje por búsqueda de políticas.

> Mensaje clave del día: una recompensa mal diseñada puede producir comportamientos extraños, eficientes en simulación, pero inválidos o peligrosos en el mundo real.


## 0. Instalación recomendada

Este cuaderno está pensado para ejecutarse en un entorno de Python con `gymnasium`, `matplotlib`, `numpy` e `ipywidgets`.

Si trabajas en **Colab** o en un entorno nuevo, ejecuta la celda de instalación. Si ya tienes los paquetes, puedes saltarla.

**Nota práctica:** MuJoCo y Box2D pueden requerir dependencias adicionales dependiendo del sistema operativo. Por eso el cuaderno tiene respaldos y mensajes de diagnóstico.

In [ ]:
# Ejecuta esta celda solo si necesitas instalar dependencias.
# En Colab suele funcionar. En Windows, Box2D puede requerir instalación adicional de swig.

# %pip install -q "gymnasium[classic-control]" "gymnasium[mujoco]" "gymnasium[box2d]" ipywidgets matplotlib pandas numpy

## 1. Importar librerías y utilidades base

Estas funciones nos permitirán:

- crear entornos de forma segura;
- correr episodios;
- comparar políticas;
- graficar recompensas;
- mostrar animaciones cuando el entorno permita renderizado.

In [ ]:
import math
import time
import warnings
from dataclasses import dataclass
from typing import Callable, Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib

try:
    from IPython import get_ipython
    if get_ipython() is None:
        matplotlib.use("Agg")
except Exception:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    import gymnasium as gym
    GYMNASIUM_OK = True
except Exception as e:
    GYMNASIUM_OK = False
    GYMNASIUM_IMPORT_ERROR = e

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, Markdown, clear_output
    WIDGETS_OK = True
except Exception as e:
    WIDGETS_OK = False
    WIDGETS_IMPORT_ERROR = e

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

if not GYMNASIUM_OK:
    print("No se pudo importar gymnasium.")
    print("Ejecuta la celda de instalación y reinicia el kernel si hace falta.")
    print("Error:", GYMNASIUM_IMPORT_ERROR)
else:
    print("Gymnasium importado correctamente.")

if not WIDGETS_OK:
    print("ipywidgets no está disponible. Las actividades interactivas funcionarán como funciones normales.")

In [ ]:
def make_env_safely(env_id: str, render_mode: Optional[str] = None):
    """Crear un entorno y devolver (env, error). Si falla, env será None."""
    if not GYMNASIUM_OK:
        return None, RuntimeError("gymnasium no está instalado")
    try:
        env = gym.make(env_id, render_mode=render_mode)
        return env, None
    except Exception as e:
        return None, e


def run_episode(env_id: str,
                policy_fn: Callable[[np.ndarray], object],
                seed: int = 0,
                max_steps: int = 500,
                render: bool = False):
    """Ejecuta un episodio y devuelve recompensa total, pasos y frames opcionales."""
    env, err = make_env_safely(env_id, render_mode="rgb_array" if render else None)
    if err is not None:
        raise RuntimeError(f"No se pudo crear el entorno {env_id}: {err}")

    obs, info = env.reset(seed=seed)
    total_reward = 0.0
    frames = []

    for t in range(max_steps):
        if render:
            try:
                frames.append(env.render())
            except Exception:
                pass

        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)

        if terminated or truncated:
            break

    env.close()
    return total_reward, t + 1, frames


def random_policy_factory(env_id: str):
    """Crea una política aleatoria compatible con el espacio de acciones del entorno."""
    env, err = make_env_safely(env_id)
    if err is not None:
        raise RuntimeError(f"No se pudo crear {env_id}: {err}")
    action_space = env.action_space
    env.close()

    def policy(obs):
        return action_space.sample()
    return policy


def evaluate_policy(env_id: str,
                    policy_fn: Callable[[np.ndarray], object],
                    episodes: int = 10,
                    max_steps: int = 500,
                    seed: int = 0) -> pd.DataFrame:
    results = []
    for ep in range(episodes):
        total, steps, _ = run_episode(env_id, policy_fn, seed=seed + ep, max_steps=max_steps, render=False)
        results.append({"episodio": ep, "recompensa": total, "pasos": steps})
    return pd.DataFrame(results)


def plot_rewards(df: pd.DataFrame, title: str = "Recompensa por episodio"):
    fig, ax = plt.subplots()
    ax.plot(df["episodio"], df["recompensa"], marker="o", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    plt.show()


def moving_average(x, window=50):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")


def score_policy(env_id: str,
                 policy_fn: Callable[[np.ndarray], object],
                 episodes: int = 3,
                 max_steps: int = 500,
                 seed: int = 0) -> float:
    """Devuelve la recompensa promedio de una política."""
    df = evaluate_policy(env_id, policy_fn, episodes=episodes, max_steps=max_steps, seed=seed)
    return float(df["recompensa"].mean())


from matplotlib import animation


def show_episode_animation(env_id,
                           policy_fn,
                           seed=0,
                           max_steps=300,
                           fps=30,
                           title=None,
                           frame_skip=2):
    """Muestra una animación corta de una política.

    `frame_skip` reduce el peso del HTML sin cambiar la simulación original.
    """
    if policy_fn is None:
        print("No hay política disponible para animar.")
        return

    try:
        if title:
            display(Markdown(f"**{title}**"))

        total, steps, frames = run_episode(env_id, policy_fn, seed=seed, max_steps=max_steps, render=True)
        if not frames:
            print("No se obtuvieron frames. El renderizado puede no estar disponible.")
            return

        if frame_skip and frame_skip > 1 and len(frames) > 2:
            reduced_frames = frames[::frame_skip]
            if reduced_frames[-1] is not frames[-1]:
                reduced_frames.append(frames[-1])
            frames = reduced_frames

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.axis("off")
        im = ax.imshow(frames[0])

        def update(i):
            im.set_array(frames[i])
            return [im]

        anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
        plt.close(fig)
        display(HTML(anim.to_jshtml()))
        print(f"Recompensa total: {total:.2f} | pasos simulados: {steps} | frames mostrados: {len(frames)}")
    except Exception as e:
        print("No se pudo generar la animación.")
        print("Motivo:", e)


def show_policy_videos(env_id, policy_specs, seed=0, max_steps=300, fps=30, frame_skip=2):
    """Muestra videos consecutivos para una lista de (titulo, política)."""
    for idx, (label, policy_fn) in enumerate(policy_specs, start=1):
        show_episode_animation(
            env_id,
            policy_fn,
            seed=seed + idx - 1,
            max_steps=max_steps,
            fps=fps,
            title=f"{idx}. {label}",
            frame_skip=frame_skip,
        )


## 2. Conceptos mínimos antes de jugar

Un problema de RL se describe con un ciclo:

- **Agente:** quien decide.
- **Entorno:** aquello con lo que interactúa.
- **Estado u observación:** lo que el agente puede percibir.
- **Acción:** lo que puede hacer.
- **Recompensa:** señal numérica que orienta el aprendizaje.
- **Política:** regla que decide qué acción tomar en cada situación.
- **Episodio:** intento completo desde un inicio hasta una condición de finalización.

La diferencia clave con aprendizaje supervisado es esta:

> No le damos al agente la respuesta correcta paso a paso; le damos consecuencias.

### Pregunta rápida

Antes de correr código, responde:

1. ¿Qué sería una acción en un robot seguidor de línea?
2. ¿Qué podría ser una recompensa útil?
3. ¿Qué recompensa mal diseñada podría producir un comportamiento indeseado?

## 3. Demo guiada — CartPole tipo MuJoCo: `InvertedPendulum-v5`

Antes de entrenar nada, conviene mirar un problema de control lo más simple posible. Aqué el agente intenta mantener un péndulo invertido en equilibrio. Esa idea aparece en robótica, drones, balanceadores y sistemas donde una desviación pequeña puede crecer rápido.

Primero intentaremos usar `InvertedPendulum-v5`, un entorno de MuJoCo parecido conceptualmente a CartPole, pero con acción continua. Si MuJoCo no está disponible, usaremos `CartPole-v1` como respaldo para que la actividad no se bloquee.

Qué observar:

- La **política aleatoria** casi no tiene información: actúa sin mirar el estado.
- La **heurística PD** usa una regla de control clásica: corrige según posición, ángulo y velocidades.
- La **política aprendida** ajustará una política lineal con CEM, un método de búsqueda de políticas útil cuando la política tiene pocos parámetros.

**Idea de la demo:** comparar intuición, control manual y aprendizaje por recompensa antes de pasar a ejercicios con más estructura.


In [ ]:
def select_inverted_pendulum_env():
    candidates = ["InvertedPendulum-v5", "InvertedPendulum-v4", "CartPole-v1"]
    diagnostics = []
    for env_id in candidates:
        env, err = make_env_safely(env_id)
        if err is None:
            env.close()
            print(f"Entorno seleccionado: {env_id}")
            return env_id, diagnostics
        diagnostics.append((env_id, str(err)))
    print("No se pudo crear ningún entorno tipo péndulo/cartpole.")
    for env_id, msg in diagnostics:
        print(f"- {env_id}: {msg}")
    return None, diagnostics

pendulum_env_id, pendulum_diagnostics = select_inverted_pendulum_env()

In [ ]:
def inverted_pendulum_pd_policy(kp=8.0, kd=1.5, kx=0.5, kv=0.2, sign=1.0, force_limit=3.0):
    """Política tipo PD para InvertedPendulum. Puede requerir ajustar signo y ganancias."""
    def policy(obs):
        obs = np.asarray(obs, dtype=float)
        x = obs[0]
        theta = obs[1]
        x_dot = obs[2]
        theta_dot = obs[3]
        u = sign * (kp * theta + kd * theta_dot + kx * x + kv * x_dot)
        return np.array([np.clip(u, -force_limit, force_limit)], dtype=np.float32)
    return policy


def cartpole_simple_policy():
    """Política muy simple para CartPole-v1: empuja hacia donde cae el poste."""
    def policy(obs):
        x, x_dot, theta, theta_dot = obs
        score = theta + 0.7 * theta_dot + 0.01 * x + 0.1 * x_dot
        return int(score > 0)
    return policy


def policy_for_selected_pendulum(kp=8.0, kd=1.5, kx=0.5, kv=0.2, sign=1.0):
    if pendulum_env_id is None:
        return None
    if "InvertedPendulum" in pendulum_env_id:
        return inverted_pendulum_pd_policy(kp=kp, kd=kd, kx=kx, kv=kv, sign=sign)
    return cartpole_simple_policy()


def _pendulum_linear_policy_from_params(params, action_space):
    params = np.asarray(params, dtype=float)
    weights = params[:-1]
    bias = params[-1]
    is_discrete = hasattr(action_space, "n")

    if is_discrete:
        def policy(obs):
            obs = np.asarray(obs, dtype=float)
            return int(np.dot(weights[: len(obs)], obs) + bias > 0.0)
    else:
        low = np.asarray(action_space.low, dtype=np.float32)
        high = np.asarray(action_space.high, dtype=np.float32)

        def policy(obs):
            obs = np.asarray(obs, dtype=float)
            raw = np.array([np.dot(weights[: len(obs)], obs) + bias], dtype=np.float32)
            return np.clip(raw, low, high)

    return policy


def learn_pendulum_policy_cem(iterations=8,
                              population=10,
                              elite_fraction=0.3,
                              episodes_per_candidate=1,
                              max_steps=300,
                              seed=42):
    """Aprende una política lineal pequeña con Cross-Entropy Method (CEM)."""
    if pendulum_env_id is None:
        return None, pd.DataFrame(), None

    env, err = make_env_safely(pendulum_env_id)
    if err is not None:
        print(f"No se pudo crear {pendulum_env_id}: {err}")
        return None, pd.DataFrame(), None

    obs_dim = int(np.prod(env.observation_space.shape))
    action_space = env.action_space
    env.close()

    if "InvertedPendulum" in pendulum_env_id:
        mean = np.array([0.5, 8.0, 0.2, 1.5, 0.0], dtype=float)
        sigma = np.array([1.0, 5.0, 1.0, 2.0, 0.2], dtype=float)
    else:
        mean = np.array([0.01, 0.1, 1.0, 0.7, 0.0], dtype=float)
        sigma = np.array([0.2, 0.5, 1.0, 1.0, 0.2], dtype=float)

    expected = obs_dim + 1
    if len(mean) != expected:
        mean = np.resize(mean, expected)
        sigma = np.resize(sigma, expected)

    rng = np.random.default_rng(seed)
    elite_count = max(2, int(population * elite_fraction))
    history = []
    best_params = mean.copy()
    best_score = -np.inf

    for iteration in range(iterations):
        candidates = [mean]
        candidates.extend(rng.normal(mean, sigma, size=(population - 1, expected)))
        scored = []

        for candidate_idx, params in enumerate(candidates):
            policy = _pendulum_linear_policy_from_params(params, action_space)
            score = score_policy(
                pendulum_env_id,
                policy,
                episodes=episodes_per_candidate,
                max_steps=max_steps,
                seed=seed + 100 * iteration + candidate_idx,
            )
            scored.append((score, np.asarray(params, dtype=float)))
            if score > best_score:
                best_score = score
                best_params = np.asarray(params, dtype=float).copy()

        scored.sort(key=lambda item: item[0], reverse=True)
        elites = np.array([params for _, params in scored[:elite_count]])
        mean = elites.mean(axis=0)
        sigma = np.maximum(elites.std(axis=0) * 0.8, 0.05)

        history.append({
            "iteración": iteration,
            "mejor_recompensa": scored[0][0],
            "promedio_elite": float(np.mean([score for score, _ in scored[:elite_count]])),
        })

    learned_policy = _pendulum_linear_policy_from_params(best_params, action_space)
    return learned_policy, pd.DataFrame(history), best_params


In [ ]:
def demo_pendulo(kp=8.0, kd=1.5, kx=0.5, kv=0.2, sign=1.0, episodios=3, max_steps=300, semilla=10):
    if pendulum_env_id is None:
        print("No hay entorno disponible. Revisa la instalación de Gymnasium.")
        return

    t0 = time.perf_counter()
    print(f"Entorno usado: {pendulum_env_id}")
    print(f"Ejecutando {episodios} episodios por política, máximo {max_steps} pasos por episodio.")
    random_policy = random_policy_factory(pendulum_env_id)
    guided_policy = policy_for_selected_pendulum(kp=kp, kd=kd, kx=kx, kv=kv, sign=sign)

    df_random = evaluate_policy(pendulum_env_id, random_policy, episodes=episodios, max_steps=max_steps, seed=semilla)
    df_guided = evaluate_policy(pendulum_env_id, guided_policy, episodes=episodios, max_steps=max_steps, seed=semilla)

    summary = pd.DataFrame({
        "política": ["aleatoria", "heurística simple"],
        "recompensa_promedio": [df_random["recompensa"].mean(), df_guided["recompensa"].mean()],
        "pasos_promedio": [df_random["pasos"].mean(), df_guided["pasos"].mean()],
    })
    display(summary)

    fig, ax = plt.subplots()
    ax.plot(df_random["episodio"], df_random["recompensa"], marker="o", label="aleatoria")
    ax.plot(df_guided["episodio"], df_guided["recompensa"], marker="o", label="heurística simple")
    ax.set_title("Comparación de políticas")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    ax.legend()
    plt.show()
    print(f"Tiempo de ejecución: {time.perf_counter() - t0:.2f} s")

if WIDGETS_OK:
    display(widgets.interactive(
        demo_pendulo,
        {"manual": True, "manual_name": "Ejecutar demo"},
        kp=widgets.FloatSlider(value=8.0, min=0.0, max=30.0, step=0.5, description="kp", continuous_update=False),
        kd=widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description="kd", continuous_update=False),
        kx=widgets.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description="kx", continuous_update=False),
        kv=widgets.FloatSlider(value=0.2, min=0.0, max=5.0, step=0.1, description="kv", continuous_update=False),
        sign=widgets.Dropdown(options=[1.0, -1.0], value=1.0, description="signo"),
        episodios=widgets.IntSlider(value=3, min=1, max=10, step=1, description="episodios", continuous_update=False),
        max_steps=widgets.IntSlider(value=300, min=50, max=500, step=50, description="max_steps", continuous_update=False),
        semilla=widgets.IntSlider(value=10, min=0, max=100, step=1, description="semilla", continuous_update=False)
    ))
else:
    demo_pendulo()

### Videos de la demo: aleatoria, heurística y aprendida

Ejecuta esta celda para ver tres comportamientos consecutivos sobre el mismo tipo de problema. El orden importa: primero miramos una línea base sin inteligencia, luego una regla diseñada a mano, y al final una política cuyos parámetros fueron ajustados por recompensa.

La política aprendida usa **Cross-Entropy Method (CEM)** sobre una política lineal. No es aprendizaje profundo: es una forma compacta y rápida de aprender una regla de control para un problema pequeño.


In [ ]:
if pendulum_env_id is None:
    print("No hay entorno de péndulo disponible para animar.")
else:
    pendulum_learned_policy, pendulum_cem_history, pendulum_learned_params = learn_pendulum_policy_cem(
        iterations=8,
        population=10,
        episodes_per_candidate=1,
        max_steps=300,
        seed=42,
    )
    display(pendulum_cem_history.tail())

    show_policy_videos(
        pendulum_env_id,
        [
            ("Política aleatoria", random_policy_factory(pendulum_env_id)),
            ("Política heurística (PD o CartPole simple)", policy_for_selected_pendulum()),
            ("Política aprendida (CEM sobre política lineal)", pendulum_learned_policy),
        ],
        seed=30,
        max_steps=180,
        fps=30,
        frame_skip=2,
    )


### Reflexión breve sobre la demo

1. ¿La política simple mejora frente a la aleatoria?
2. ¿Qué ocurre cuando cambias las ganancias o el signo?
3. ¿Qué tendría que pasar para llamar esto “control confiable” y no solo “una heurística que a veces funciona”?
4. ¿Qué relación ves entre esta demo y el debate entre control clásico y RL?

## 4. Ejercicio 1 — `MountainCar-v0`: recompensa diferida

En `MountainCar`, el carro está atrapado en un valle. Para llegar a la cima, muchas veces debe moverse primero en sentido contrario y ganar energía. Es un ejemplo pequeño, pero captura una dificultad enorme de RL: una acción que parece mala ahora puede ser necesaria para ganar después.

Este entorno es perfecto para ver tres ideas:

- **Exploración:** si el agente no prueba acciones raras, nunca descubre cómo salir del valle.
- **Recompensa diferida:** el premio útil llega tarde, no en el primer paso.
- **Representación del estado:** discretizar demasiado borra detalles; discretizar demasiado fino puede hacer lento el aprendizaje.

Usaremos una versión sencilla de **Q-learning tabular con discretización** para entender la actualización de valores. Para los videos comparativos, además, usaremos CEM para aprender una política compacta de dos acciones; esto permite ver una política aprendida sin esperar un entrenamiento largo.


In [ ]:
mountain_env_id = "MountainCar-v0"
mountain_env, mountain_err = make_env_safely(mountain_env_id)
if mountain_err is None:
    print(f"{mountain_env_id} disponible.")
    print("Observación:", mountain_env.observation_space)
    print("Acciones:", mountain_env.action_space)
    mountain_env.close()
else:
    print(f"No se pudo crear {mountain_env_id}: {mountain_err}")

In [ ]:
@dataclass
class Discretizer:
    low: np.ndarray
    high: np.ndarray
    bins: Tuple[int, int]

    def transform(self, obs):
        obs = np.asarray(obs, dtype=float)
        ratios = (obs - self.low) / (self.high - self.low)
        ratios = np.clip(ratios, 0, 0.999999)
        indices = (ratios * np.array(self.bins)).astype(int)
        return tuple(indices)


def train_mountaincar_qlearning(episodes=2000,
                                bins=20,
                                alpha=0.1,
                                gamma=0.99,
                                epsilon_start=1.0,
                                epsilon_end=0.05,
                                reward_shaping=False,
                                seed=0):
    env, err = make_env_safely(mountain_env_id)
    if err is not None:
        raise RuntimeError(f"No se pudo crear {mountain_env_id}: {err}")

    low = env.observation_space.low
    high = env.observation_space.high
    discretizer = Discretizer(low=low, high=high, bins=(bins, bins))
    n_actions = env.action_space.n
    Q = np.zeros((bins, bins, n_actions), dtype=float)
    rewards = []

    rng = np.random.default_rng(seed)

    for ep in range(episodes):
        obs, info = env.reset(seed=seed + ep)
        state = discretizer.transform(obs)
        total_reward = 0.0
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * math.exp(-4 * ep / max(1, episodes))

        for t in range(200):
            if rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_obs, reward, terminated, truncated, info = env.step(action)

            # Reward shaping opcional: ayuda a visualizar el efecto de cambiar la señal de recompensa.
            # No es "gratis": modifica el problema y puede sesgar el comportamiento.
            shaped = float(reward)
            if reward_shaping:
                position, velocity = next_obs
                shaped += 0.2 * (position + 0.5) + 2.0 * abs(velocity)

            next_state = discretizer.transform(next_obs)
            best_next = np.max(Q[next_state])
            Q[state + (action,)] += alpha * (shaped + gamma * best_next - Q[state + (action,)])

            obs = next_obs
            state = next_state
            total_reward += float(reward)

            if terminated or truncated:
                break

        rewards.append(total_reward)

    env.close()
    return Q, discretizer, np.array(rewards)


def mountaincar_policy_from_Q(Q, discretizer):
    def policy(obs):
        state = discretizer.transform(obs)
        return int(np.argmax(Q[state]))
    return policy


def mountaincar_momentum_policy():
    """Heurística: empujar en el sentido de la velocidad para acumular energía."""
    def policy(obs):
        position, velocity = obs
        return 2 if velocity >= 0 else 0
    return policy


def mountaincar_policy_from_params(params):
    """Política compacta para CEM: decide izquierda/derecha con una frontera lineal."""
    params = np.asarray(params, dtype=float)

    def policy(obs):
        position, velocity = obs
        score = params[0] * (position + 0.5) + params[1] * velocity + params[2]
        return 2 if score > 0.0 else 0

    return policy


def learn_mountaincar_policy_cem(iterations=10,
                                 population=14,
                                 elite_fraction=0.3,
                                 episodes_per_candidate=2,
                                 max_steps=200,
                                 seed=7):
    """Aprende una política pequeña con CEM para producir un video comparativo rápido."""
    if mountain_err is not None:
        return None, pd.DataFrame(), None

    rng = np.random.default_rng(seed)
    mean = np.array([0.0, 10.0, 0.0], dtype=float)
    sigma = np.array([2.0, 20.0, 0.5], dtype=float)
    elite_count = max(2, int(population * elite_fraction))
    history = []
    best_params = mean.copy()
    best_score = -np.inf

    for iteration in range(iterations):
        candidates = [mean]
        candidates.extend(rng.normal(mean, sigma, size=(population - 1, 3)))
        scored = []

        for candidate_idx, params in enumerate(candidates):
            policy = mountaincar_policy_from_params(params)
            score = score_policy(
                mountain_env_id,
                policy,
                episodes=episodes_per_candidate,
                max_steps=max_steps,
                seed=seed + 100 * iteration + candidate_idx,
            )
            scored.append((score, np.asarray(params, dtype=float)))
            if score > best_score:
                best_score = score
                best_params = np.asarray(params, dtype=float).copy()

        scored.sort(key=lambda item: item[0], reverse=True)
        elites = np.array([params for _, params in scored[:elite_count]])
        mean = elites.mean(axis=0)
        sigma = np.maximum(elites.std(axis=0) * 0.8, np.array([0.03, 0.1, 0.005]))

        history.append({
            "iteración": iteration,
            "mejor_recompensa": scored[0][0],
            "promedio_elite": float(np.mean([score for score, _ in scored[:elite_count]])),
        })

    return mountaincar_policy_from_params(best_params), pd.DataFrame(history), best_params


In [ ]:
def demo_mountaincar(episodes=1500, bins=20, alpha=0.1, gamma=0.99,
                     epsilon_start=1.0, epsilon_end=0.05, reward_shaping=False, seed=0):
    if mountain_err is not None:
        print("MountainCar no está disponible. Revisa la instalación de gymnasium[classic-control].")
        return

    Q, discretizer, rewards = train_mountaincar_qlearning(
        episodes=episodes,
        bins=bins,
        alpha=alpha,
        gamma=gamma,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        reward_shaping=reward_shaping,
        seed=seed,
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(rewards, alpha=0.25, label="recompensa por episodio")
    ma = moving_average(rewards, window=max(20, episodes // 40))
    ax.plot(np.arange(len(ma)) + max(20, episodes // 40) - 1, ma, label="promedio móvil")
    ax.set_title("Entrenamiento Q-learning en MountainCar")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    ax.legend()
    plt.show()

    policy = mountaincar_policy_from_Q(Q, discretizer)
    df_eval = evaluate_policy(mountain_env_id, policy, episodes=10, max_steps=200, seed=seed + 10000)
    display(df_eval.describe()[["recompensa", "pasos"]])

    print("Lectura rápida:")
    print("- En MountainCar, recompensas más cercanas a 0 suelen ser mejores.")
    print("- Si la curva no mejora mucho, no significa que RL no sirva; significa que el problema, discretización o recompensa pueden requerir ajuste.")
    print("- Si activas reward_shaping, estás cambiando la señal que guía el aprendizaje: úsalo con criterio.")

if WIDGETS_OK:
    display(widgets.interactive(
        demo_mountaincar,
        {"manual": True, "manual_name": "Entrenar Q-learning"},
        episodes=widgets.IntSlider(value=1500, min=200, max=6000, step=100, description="episodios", continuous_update=False),
        bins=widgets.IntSlider(value=20, min=8, max=40, step=2, description="bins", continuous_update=False),
        alpha=widgets.FloatSlider(value=0.1, min=0.01, max=0.8, step=0.01, description="alpha", continuous_update=False),
        gamma=widgets.FloatSlider(value=0.99, min=0.80, max=0.999, step=0.001, description="gamma", continuous_update=False),
        epsilon_start=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="eps inicial", continuous_update=False),
        epsilon_end=widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description="eps final", continuous_update=False),
        reward_shaping=widgets.Checkbox(value=False, description="reward shaping"),
        seed=widgets.IntSlider(value=0, min=0, max=100, step=1, description="semilla", continuous_update=False)
    ))
else:
    demo_mountaincar()

### Videos de MountainCar: aleatoria, heurística y aprendida

Aqué los tres videos ayudan a ver la recompensa diferida. La política aleatoria suele quedarse explorando sin propósito. La heurística empuja según la velocidad para acumular energía. La política aprendida usa CEM para ajustar una frontera de decisión pequeña: cuándo empujar a la izquierda y cuándo a la derecha.

El Q-learning de la celda anterior sigue siendo el ejercicio principal para entender valores `Q(s, a)`. Este video usa CEM porque produce una política visual en segundos y mantiene la comparación pedagógica clara.


In [ ]:
if mountain_err is not None:
    print("MountainCar no está disponible para animar.")
else:
    mountain_learned_policy, mountain_cem_history, mountain_learned_params = learn_mountaincar_policy_cem(
        iterations=10,
        population=14,
        episodes_per_candidate=2,
        max_steps=200,
        seed=7,
    )
    display(mountain_cem_history.tail())

    show_policy_videos(
        mountain_env_id,
        [
            ("Política aleatoria", random_policy_factory(mountain_env_id)),
            ("Política heurística de momento", mountaincar_momentum_policy()),
            ("Política aprendida (CEM sobre frontera lineal)", mountain_learned_policy),
        ],
        seed=40,
        max_steps=200,
        fps=30,
        frame_skip=2,
    )


### Preguntas sobre MountainCar

1. ¿Qué pasa si reduces mucho la exploración?
2. ¿Qué cambia al activar `reward_shaping`?
3. ¿Por qué una recompensa intermedia puede acelerar el aprendizaje, pero también sesgarlo?
4. ¿Qué analogía encuentras entre MountainCar y decisiones reales donde toca sacrificar el corto plazo?

## 5. Ejercicio 2 — `LunarLander-v3`: aterrizar con criterio

`LunarLander` es más visual y tiene más sabor de control. El objetivo no es solo tocar el suelo: queremos aterrizar de forma estable, con poco combustible, buen ángulo y velocidad razonable.

Antes de mirar recompensas, piensa en el conflicto:

- Si corriges mucho el ángulo, puedes gastar demasiado o generar oscilaciones.
- Si bajas muy rápido, ganas tiempo pero arriesgas choque.
- Si te quedas flotando, controlas mejor pero pagas combustible.

Aqué compararemos:

- una política aleatoria;
- una heurística simple basada en posición, velocidad, ángulo y contacto;
- una política aprendida con CEM que ajusta automáticamente los parámetros de esa heurística usando recompensa.

La idea es observar que una política debe balancear varios objetivos a la vez, y que aprender no significa magia: significa buscar parámetros que producen mejores consecuencias.


In [ ]:
def select_lunar_env():
    candidates = ["LunarLander-v3", "LunarLander-v2"]
    diagnostics = []
    for env_id in candidates:
        env, err = make_env_safely(env_id)
        if err is None:
            env.close()
            print(f"Entorno seleccionado: {env_id}")
            return env_id, diagnostics
        diagnostics.append((env_id, str(err)))
    print("No se pudo crear LunarLander. Probablemente falta Box2D o swig.")
    for env_id, msg in diagnostics:
        print(f"- {env_id}: {msg}")
    return None, diagnostics

lunar_env_id, lunar_diagnostics = select_lunar_env()

In [ ]:
def lunar_heuristic_policy(angle_gain=0.5, angle_vel_gain=1.0, hover_gain=0.5, vel_gain=0.5, x_gain=0.5, vx_gain=1.0):
    """Heurística simple para LunarLander discreto.

    Acción típica:
    0: nada, 1: motor lateral izquierdo, 2: motor principal, 3: motor lateral derecho.
    No es una solución perfecta; sirve para comparar contra aleatorio y discutir diseño de políticas.
    """
    def policy(obs):
        x, y, vx, vy, angle, angular_vel, left_contact, right_contact = obs

        angle_target = x_gain * x + vx_gain * vx
        angle_target = np.clip(angle_target, -0.4, 0.4)
        hover_target = 0.55 * abs(x)

        angle_todo = angle_gain * (angle_target - angle) - angle_vel_gain * angular_vel
        hover_todo = hover_gain * (hover_target - y) - vel_gain * vy

        if left_contact or right_contact:
            angle_todo = 0.0
            hover_todo = -vel_gain * vy

        if hover_todo > abs(angle_todo) and hover_todo > 0.05:
            return 2
        if angle_todo < -0.05:
            return 3
        if angle_todo > 0.05:
            return 1
        return 0
    return policy


def demo_lunar(angle_gain=0.5, angle_vel_gain=1.0, hover_gain=0.5, vel_gain=0.5, x_gain=0.5, vx_gain=1.0, episodios=5, max_steps=600, seed=20):
    if lunar_env_id is None:
        print("LunarLander no está disponible en este entorno.")
        print("En Colab, intenta ejecutar la celda de instalación con gymnasium[box2d].")
        return

    random_policy = random_policy_factory(lunar_env_id)
    heuristic_policy = lunar_heuristic_policy(
        angle_gain=angle_gain,
        angle_vel_gain=angle_vel_gain,
        hover_gain=hover_gain,
        vel_gain=vel_gain,
        x_gain=x_gain,
        vx_gain=vx_gain,
    )

    df_random = evaluate_policy(lunar_env_id, random_policy, episodes=episodios, max_steps=max_steps, seed=seed)
    df_heur = evaluate_policy(lunar_env_id, heuristic_policy, episodes=episodios, max_steps=max_steps, seed=seed)

    summary = pd.DataFrame({
        "política": ["aleatoria", "heurística simple"],
        "recompensa_promedio": [df_random["recompensa"].mean(), df_heur["recompensa"].mean()],
        "pasos_promedio": [df_random["pasos"].mean(), df_heur["pasos"].mean()],
    })
    display(summary)

    fig, ax = plt.subplots()
    ax.plot(df_random["episodio"], df_random["recompensa"], marker="o", label="aleatoria")
    ax.plot(df_heur["episodio"], df_heur["recompensa"], marker="o", label="heurística")
    ax.set_title("LunarLander: política aleatoria vs heurística")
    ax.set_xlabel("Episodio")
    ax.set_ylabel("Recompensa total")
    ax.legend()
    plt.show()

def lunar_policy_from_params(params):
    return lunar_heuristic_policy(*np.asarray(params, dtype=float))


def learn_lunar_policy_cem(iterations=6,
                           population=10,
                           elite_fraction=0.3,
                           episodes_per_candidate=1,
                           max_steps=600,
                           seed=30):
    """Ajusta los parámetros de la heurística de LunarLander con CEM."""
    if lunar_env_id is None:
        return None, pd.DataFrame(), None

    rng = np.random.default_rng(seed)
    mean = np.array([0.5, 1.0, 0.5, 0.5, 0.5, 1.0], dtype=float)
    sigma = np.array([0.5, 0.7, 0.5, 0.5, 0.7, 0.8], dtype=float)
    bounds = np.array([
        [0.0, 2.5],
        [0.0, 3.0],
        [0.0, 2.0],
        [0.0, 2.0],
        [-1.0, 2.5],
        [-1.0, 3.5],
    ])
    elite_count = max(2, int(population * elite_fraction))
    history = []
    best_params = mean.copy()
    best_score = -np.inf

    for iteration in range(iterations):
        candidates = [mean]
        candidates.extend(np.clip(rng.normal(mean, sigma, size=(population - 1, len(mean))), bounds[:, 0], bounds[:, 1]))
        scored = []

        for candidate_idx, params in enumerate(candidates):
            policy = lunar_policy_from_params(params)
            score = score_policy(
                lunar_env_id,
                policy,
                episodes=episodes_per_candidate,
                max_steps=max_steps,
                seed=seed + 100 * iteration + candidate_idx,
            )
            scored.append((score, np.asarray(params, dtype=float)))
            if score > best_score:
                best_score = score
                best_params = np.asarray(params, dtype=float).copy()

        scored.sort(key=lambda item: item[0], reverse=True)
        elites = np.array([params for _, params in scored[:elite_count]])
        mean = elites.mean(axis=0)
        sigma = np.maximum(elites.std(axis=0) * 0.8, 0.05)

        history.append({
            "iteración": iteration,
            "mejor_recompensa": scored[0][0],
            "promedio_elite": float(np.mean([score for score, _ in scored[:elite_count]])),
        })

    return lunar_policy_from_params(best_params), pd.DataFrame(history), best_params


if WIDGETS_OK:
    display(widgets.interactive(
        demo_lunar,
        {"manual": True, "manual_name": "Evaluar LunarLander"},
        angle_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="angle_gain", continuous_update=False),
        angle_vel_gain=widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description="ang_vel", continuous_update=False),
        hover_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="hover", continuous_update=False),
        vel_gain=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description="vel", continuous_update=False),
        x_gain=widgets.FloatSlider(value=0.5, min=-1.0, max=2.0, step=0.05, description="x_gain", continuous_update=False),
        vx_gain=widgets.FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.05, description="vx_gain", continuous_update=False),
        episodios=widgets.IntSlider(value=5, min=1, max=20, step=1, description="episodios", continuous_update=False),
        max_steps=widgets.IntSlider(value=600, min=100, max=1000, step=100, description="max_steps", continuous_update=False),
        seed=widgets.IntSlider(value=20, min=0, max=100, step=1, description="semilla", continuous_update=False)
    ))
else:
    demo_lunar()

### Videos de LunarLander: aleatoria, heurística y aprendida

En este bloque conviene mirar el video como si fuera una prueba de control: orientación, velocidad, combustible implícito y contacto con el suelo. La política aleatoria muestra el caos inicial; la heurística muestra conocimiento humano; la aprendida ajusta los pesos de esa heurística con recompensa.

El método usado es nuevamente **CEM**, ahora sobre los parámetros de la heurística. Es una forma útil de conectar ingeniería de control con aprendizaje por refuerzo: no reemplaza la intuición, la convierte en un punto de partida optimizable.


In [ ]:
if lunar_env_id is None:
    print("LunarLander no está disponible para animar.")
else:
    lunar_learned_policy, lunar_cem_history, lunar_learned_params = learn_lunar_policy_cem(
        iterations=6,
        population=10,
        episodes_per_candidate=1,
        max_steps=600,
        seed=30,
    )
    display(lunar_cem_history.tail())
    display(pd.DataFrame({
        "parámetro": ["angle_gain", "angle_vel_gain", "hover_gain", "vel_gain", "x_gain", "vx_gain"],
        "valor_aprendido": lunar_learned_params,
    }))

    show_policy_videos(
        lunar_env_id,
        [
            ("Política aleatoria", random_policy_factory(lunar_env_id)),
            ("Política heurística", lunar_heuristic_policy()),
            ("Política aprendida (CEM sobre parámetros heurísticos)", lunar_learned_policy),
        ],
        seed=50,
        max_steps=400,
        fps=30,
        frame_skip=3,
    )


### Preguntas sobre LunarLander

1. ¿Qué variables parecen más importantes para aterrizar bien?
2. ¿Qué ocurre si la heurística corrige demasiado el ángulo?
3. ¿Cómo cambiaría la recompensa si quieres aterrizajes más suaves aunque tarden más?
4. ¿Qué riesgos tendría transferir una política de simulación directamente a un dron o robot real?

## 6. Human-in-the-loop y LLMs: conexión conceptual

En RL clásico, una recompensa guía el comportamiento. En sistemas modernos, el humano puede entrar de varias formas:

- dando demostraciones;
- corrigiendo acciones;
- comparando respuestas;
- expresando preferencias;
- definiendo restricciones de seguridad.

En LLMs, esta idea aparece en métodos como RLHF: el sistema no solo aprende de texto, sino también de preferencias humanas sobre qué respuestas son más útiles, seguras o alineadas con una intención.

### Preguntas

1. ¿Qué ventaja tiene usar feedback humano?
2. ¿Qué sesgos o riesgos puede introducir?
3. ¿Por qué human-in-the-loop no vuelve automáticamente seguro a un sistema?

## 7. Cierre: formula un problema de RL

Escoge un sistema de ingeniería o mecatrónica y formula un problema de RL en términos de:

| Elemento | Tu respuesta |
|---|---|
| Agente |  |
| Entorno |  |
| Estado u observación |  |
| Acciones |  |
| Recompensa |  |
| Episodio |  |
| Riesgo de reward hacking |  |
| Restricciones de seguridad |  |
| ¿Qué simularías antes de probar en físico? |  |

### Pregunta final

¿Qué te gustaría explorar después de esta puerta de entrada: robótica, videojuegos, control, agentes, simulación, LLMs con feedback humano u otro camino?

## 8. Declaración breve de uso de IA

Si usaste Codex, ChatGPT, Gemini, Claude u otra herramienta durante este cuaderno, registra brevemente:

1. ¿Qué herramienta usaste?
2. ¿Para qué la usaste?
3. ¿Qué respuesta o fragmento te ayudó?
4. ¿Qué verificaste manualmente?
5. ¿Qué parte del razonamiento puedes defender con tus propias palabras?